In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t083/task.json'

with open(task_json_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "frontend异常告警" and occurred within the time window from 2026-03-31 16:03:19.612752 to 2026-03-31 16:08:05 (UTC+8). The entity that triggered the alert is an APM operation named "frontend::POST" with the entity ID `c6388887e91aa969bdf210fe96a48098`. The user is requesting root cause analysis for this alert.

The original code execution output of IPython Kernel is also provided below for reference:

('frontend异常告警', {'start': '2026-03-31T16:03:19.612752+08:00', 'end': '2026-03-31T16:08:05+08:00'}, {'entity_id': 'c6388887e91aa969bdf210fe96a48098', 'entity_name': 'frontend::POST', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">frontend异常告警<alert_event event_id="69158ae42ad2d9f290bed818d81e1743" trans_id="nku651fobh73722ha4usui9ca7" rule_id="d4107ac6-2a37-4167-9059-16156c90f2bf" rule_name="frontend异常告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t083/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# Convert 'time' and 'value' columns to numeric types
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter for the 'frontend' service (entity_set='apm.service.legacy')
frontend_metrics = metrics_df[(metrics_df['entity_set'] == 'apm.service.legacy') & 
                               (metrics_df['entity_name'] == 'frontend')]

# Calculate global thresholds (e.g., P95) for each KPI
global_thresholds = frontend_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['metric', 'p95_threshold']

global_thresholds

```
Out[2]:
```
The global P95 thresholds for the KPIs of the 'frontend' service are as follows:
- CPU: 0.3344088
- Error: 1114.3
- Latency: 0.09556603
- Memory: 267,123,000
- Slow: 2209.1
- Workload: 44,190.25

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0       cpu   3.344088e-01
1     error   1.114300e+03
2   latency   9.556603e-02
3       mem   2.671230e+08
4      slow   2.209100e+03
5  workload   4.419025e+04```
```

In [ ]:
# In[3]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-03-31T16:03:19.612752+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-03-31T16:08:05+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert microseconds to seconds for comparison
frontend_metrics['time_seconds'] = frontend_metrics['time'] / 1_000_000

# Filter metrics within the alert time window
filtered_metrics = frontend_metrics[
    (frontend_metrics['time_seconds'] >= alert_start.timestamp()) &
    (frontend_metrics['time_seconds'] <= alert_end.timestamp())
]

# Merge with global thresholds to check for anomalies
anomalies = filtered_metrics.merge(global_thresholds, on='metric')
anomalies['is_anomalous'] = anomalies['value'] > anomalies['p95_threshold']

# Filter only anomalous data
anomalous_data = anomalies[anomalies['is_anomalous']]

anomalous_data

```
Out[3]:
```
During the alert time window (2026-03-31 16:03:19.612752 to 2026-03-31 16:08:05 UTC+8), the 'frontend' service exhibited an anomaly in the "error" KPI. The observed value exceeded the global P95 threshold of 1114.3, indicating abnormal behavior.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id entity_name  ... metric_set_id  service  time_seconds p95_threshold  is_anomalous
19  1774944319000000    apm  apm.service.legacy              frontend  ...          None     None  1.774944e+09        1114.3          True

[1 rows x 12 columns]```
```

In [ ]:
# In[4]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t083/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Convert 'startTime', 'endTime', and 'duration' to numeric types
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Convert nanoseconds to seconds for time comparison
traces_df['startTime_seconds'] = traces_df['startTime'] / 1_000_000_000
traces_df['endTime_seconds'] = traces_df['endTime'] / 1_000_000_000

# Filter traces within the alert time window and related to the 'frontend' service
filtered_traces = traces_df[
    (traces_df['startTime_seconds'] >= alert_start.timestamp()) &
    (traces_df['endTime_seconds'] <= alert_end.timestamp()) &
    (traces_df['serviceName'] == 'frontend')
]

# Analyze traces to identify if the issue propagates from an upstream service
# Check for parent spans to see if the issue originates upstream
upstream_analysis = filtered_traces[['traceId', 'spanId', 'parentSpanId', 'serviceName', 'spanName', 'statusCode', 'statusMessage']]

filtered_traces, upstream_analysis

```
Out[4]:
```
The traces within the alert time window show that the 'frontend' service is involved in multiple operations, including calls to upstream services like `ProductCatalogService` and `CartService`. Some of these upstream calls, such as `grpc.oteldemo.CartService/AddItem`, have a non-zero `statusCode` (e.g., 2), indicating errors. This suggests that the issue may propagate from upstream services, particularly `CartService`, rather than originating solely within the 'frontend' service. Further investigation into the upstream services is needed to confirm the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 traceId            spanId      parentSpanId kind                                           spanName  ...  events    ip  links startTime_seconds endTime_seconds
399590  e76d026a215212d4e0c1bfa3d87e7ec4  a5cfabe532ac8e5c  ee5f9e5ab97de812    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09
399591  73ea87f6824154f37c3395e365e7bb74  5a68636fca1be95d  d3694b9b3ef103b1    3                  grpc.oteldemo.CartService/AddItem  ...    None  None   None      1.774944e+09    1.774944e+09
399592  b0e7f08eeb4dd7d3093630f81c6898e5  bda39aafa91b68cb  9b368d93714f08ee    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09
399593  39c1b392d7a4aa70e68790e117d38ad4  4b5ac68b79334b0c  baf1718fb426cff0    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09
399594  9022fbe9e26bad9f2dfaab523eb9ee40  0d7ff9374de41fa9  0293882a9412c11c    3                  grpc.oteldemo.CartService/AddItem  ...    None  None   None      1.774944e+09    1.774944e+09
...                                  ...               ...               ...  ...                                                ...  ...     ...   ...    ...               ...             ...
497877  e4472bda6d8b920f94b92a90f67bd013  73d50ba3cd2ddd58  b0b480a212818d10    3  grpc.oteldemo.RecommendationService/ListRecomm...  ...    None  None   None      1.774944e+09    1.774944e+09
497878  e4472bda6d8b920f94b92a90f67bd013  a43febfe2bf60c9a  b0b480a212818d10    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09
497879  e4472bda6d8b920f94b92a90f67bd013  f6e81bd93f4cb1dc  b0b480a212818d10    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09
497880  e4472bda6d8b920f94b92a90f67bd013  002db0edc2978734  b0b480a212818d10    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09
497881  e4472bda6d8b920f94b92a90f67bd013  9fbd56ace90d2893  b0b480a212818d10    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...    None  None   None      1.774944e+09    1.774944e+09

[38424 rows x 23 columns],                                  traceId            spanId      parentSpanId serviceName                                           spanName statusCode statusMessage
399590  e76d026a215212d4e0c1bfa3d87e7ec4  a5cfabe532ac8e5c  ee5f9e5ab97de812    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              
399591  73ea87f6824154f37c3395e365e7bb74  5a68636fca1be95d  d3694b9b3ef103b1    frontend                  grpc.oteldemo.CartService/AddItem          2              
399592  b0e7f08eeb4dd7d3093630f81c6898e5  bda39aafa91b68cb  9b368d93714f08ee    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              
399593  39c1b392d7a4aa70e68790e117d38ad4  4b5ac68b79334b0c  baf1718fb426cff0    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              
399594  9022fbe9e26bad9f2dfaab523eb9ee40  0d7ff9374de41fa9  0293882a9412c11c    frontend                  grpc.oteldemo.CartService/AddItem          2              
...                                  ...               ...               ...         ...                                                ...        ...           ...
497877  e4472bda6d8b920f94b92a90f67bd013  73d50ba3cd2ddd58  b0b480a212818d10    frontend  grpc.oteldemo.RecommendationService/ListRecomm...          0              
497878  e4472bda6d8b920f94b92a90f67bd013  a43febfe2bf60c9a  b0b480a212818d10    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              
497879  e4472bda6d8b920f94b92a90f67bd013  f6e81bd93f4cb1dc  b0b480a212818d10    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              
497880  e4472bda6d8b920f94b92a90f67bd013  002db0edc2978734  b0b480a212818d10    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              
497881  e4472bda6d8b920f94b92a90f67bd013  9fbd56ace90d2893  b0b480a212818d10    frontend     grpc.oteldemo.ProductCatalogService/GetProduct          0              

[38424 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Filter for the 'CartService' service (entity_set='apm.service.legacy')
cart_service_metrics = metrics_df[(metrics_df['entity_set'] == 'apm.service.legacy') & 
                                  (metrics_df['entity_name'] == 'cart')]

# Calculate global thresholds (e.g., P95) for each KPI
cart_service_thresholds = cart_service_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
cart_service_thresholds.columns = ['metric', 'p95_threshold']

cart_service_thresholds

```
Out[5]:
```
The global P95 thresholds for the KPIs of the `CartService` are as follows:
- CPU: 0.1783239
- Error: 557.15
- Latency: 0.004537152
- Memory: 124,661,000
- Workload: 22,051.95

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0       cpu   1.783239e-01
1     error   5.571500e+02
2   latency   4.537152e-03
3       mem   1.246610e+08
4  workload   2.205195e+04```
```

In [ ]:
# In[6]:

# Convert microseconds to seconds for comparison in the 'CartService' metrics
cart_service_metrics['time_seconds'] = cart_service_metrics['time'] / 1_000_000

# Filter metrics within the alert time window for 'CartService'
filtered_cart_metrics = cart_service_metrics[
    (cart_service_metrics['time_seconds'] >= alert_start.timestamp()) &
    (cart_service_metrics['time_seconds'] <= alert_end.timestamp())
]

# Merge with global thresholds for 'CartService' to check for anomalies
cart_anomalies = filtered_cart_metrics.merge(cart_service_thresholds, on='metric')
cart_anomalies['is_anomalous'] = cart_anomalies['value'] > cart_anomalies['p95_threshold']

# Filter only anomalous data for 'CartService'
cart_anomalous_data = cart_anomalies[cart_anomalies['is_anomalous']]

cart_anomalous_data

```
Out[6]:
```
During the alert time window (2026-03-31 16:03:19.612752 to 2026-03-31 16:08:05 UTC+8), the `CartService` exhibited multiple anomalies. These include values exceeding the global P95 thresholds for the "latency" and "error" KPIs. This indicates that the `CartService` experienced abnormal behavior during the incident, which could be contributing to the issue observed in the `frontend` service.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id entity_name  ... metric_set_id  service  time_seconds p95_threshold  is_anomalous
7   1774944259000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09      0.004537          True
12  1774944289000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09      0.004537          True
16  1774944319000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09    557.150000          True
17  1774944319000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09      0.004537          True
22  1774944349000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09      0.004537          True
27  1774944379000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09      0.004537          True
32  1774944409000000    apm  apm.service.legacy                  cart  ...          None     None  1.774944e+09      0.004537          True

[7 rows x 12 columns]```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t083/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Convert '_time_' column to numeric if necessary and filter logs within the alert time window
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs related to the 'CartService' during the alert time window
filtered_logs = logs_df[
    (logs_df['_time_'] >= alert_start.timestamp() * 1_000) &  # Convert seconds to milliseconds
    (logs_df['_time_'] <= alert_end.timestamp() * 1_000) & 
    (logs_df['_container_name_'].str.contains('cart', case=False, na=False))
]

# Analyze logs for error messages or indications of root cause reasons
error_logs = filtered_logs[filtered_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[7]:
```
No error logs related to the `CartService` were found during the alert time window (2026-03-31 16:03:19.612752 to 2026-03-31 16:08:05 UTC+8). This suggests that the root cause may not be directly evident from the logs of the `CartService`. Further investigation into other telemetry data or upstream dependencies may be required.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_]
Index: []

[0 rows x 15 columns]```
```